## II. Equilibrio de fases: Agua–mineral

#### 2.1 Agua – Halita

En esta sección estudiaremos el equilibrio entre el agua (o una solución) y un gas (principalmente CO₂). Luego, nos enfocaremos en el equilibrio entre uno o más minerales y el agua, y finalmente, examinaremos el equilibrio del agua con **tanto un gas como un mineral**.

Todas estas combinaciones se construyen a partir de una plantilla muy similar a la utilizada en la sección “**Especiación de un agua natural**”, pero en este caso usando la palabra clave `EQUILIBRIUM_PHASES`.

A lo largo de esta sección, y mediante algunos ejemplos, abordaremos al menos los siguientes problemas geoquímicos:

- Determinación del **pH de la lluvia**.
- **Solubilidad de minerales**.
- **Humedad relativa de delicuescencia**.
- **Solubilidad de carbonatos** en presencia de CO₂.
- **Estabilidad de minerales** con diferentes grados de hidratación (por ejemplo, yeso–anhidrita) y minerales polimorfos (por ejemplo, calcita–aragonito).
- Comparación entre un **sistema abierto** y un **sistema cerrado** con respecto al CO₂.


En este ejemplo, se pone en contacto agua pura con halita (**NaCl**) hasta alcanzar el equilibrio. A diferencia del caso del CO₂, cuando se trabaja con un mineral se debe definir tanto el **índice de saturación** del mineral como el **número de moles** disponibles para el cálculo. Es decir, `Halite 0 10` significa:

- Halite → el nombre del mineral,
- `0` → el índice de saturación (SI = 0, equilibrio),
- `10` → los moles disponibles para alcanzar el equilibrio.

Se recomienda usar una cantidad grande de moles para evitar problemas numéricos; PHREEQC por defecto usa 10 moles. Si el número de moles es cero, el mineral no puede disolverse, pero **sí puede precipitar**.

En este ejemplo, el índice de saturación es 0, lo que indica equilibrio con la solución. Sin embargo, se puede definir otro valor según el objetivo de la simulación. Por ejemplo, SI = 1 implica sobresaturación (el mineral tendería a precipitar), mientras que SI = –0.5 indica subsaturación (el mineral tendería a disolverse).

Los nombres de los minerales deben escribirse **en inglés**. Una forma rápida de verificar la ortografía correcta es buscar en la base de datos de PHREEQC, en la sección `PHASES`.

Este tipo de plantilla es muy útil para calcular la **solubilidad de un mineral** en agua a diferentes temperaturas, cuantificar su solubilidad en presencia de otros iones (aguas naturales) y calcular la **humedad relativa de delicuescencia**.

El archivo de salida, al igual que en el caso anterior de equilibrio con CO₂, muestra dos partes:

1. Especiación del agua pura.
2. Especiación tras alcanzar el equilibrio con halita (*Beginning of batch-reaction calculations*).

La segunda parte (`Solution composition`) calcula los moles de Na y Cl necesarios para que el índice de saturación sea igual a 0, es decir, **se calcula la solubilidad de la halita**.

En la sección `Description of solution` se obtienen parámetros interesantes como:
- pH
- conductividad eléctrica
- densidad
- fuerza iónica
- actividad del agua

En equilibrio, el **potencial químico** del agua líquida es igual al del vapor, por lo tanto, la **actividad del agua es igual a la humedad relativa en equilibrio**, también llamada **humedad relativa de delicuescencia**, y se expresa como fracción.

Anteriormente mencionamos que el equilibrio entre agua pura y halita se expresa como `Halite 0 10`. En esta situación, PHREEQC disuelve (quita iones del sólido) o precipita (añade iones al sólido) hasta alcanzar el equilibrio con la solución. Esta variación se observa en la sección *Beginning of batch-reaction calculations*:

- Un **signo negativo** indica que el mineral **se disuelve** hasta alcanzar el equilibrio.
- Un **signo positivo** indica que el mineral **precipita**.

Si se desea que la simulación permita **solo precipitación** o **solo disolución**, se debe indicar:

- `Halite 0 0` → el mineral **puede precipitar pero no disolverse**

- `halite 0 10` dis (dissolves but does not precipitate the mineral)

In [ ]:
import subprocess

# Step 1: Create the PHREEQC input file
input_file_content = """
TITLE Plantilla: Equilibrio entre el agua y el CO2.
SOLUTION 1
temp      25
EQUILIBRIUM_PHASES
Halite 0 10            
END
"""

# Save the input file
input_file_name = "species_equilibrium.pqi"
with open(input_file_name, "w") as file:
    file.write(input_file_content)
print(f"PHREEQC input file '{input_file_name}' created successfully.")

# Step 2: Run PHREEQC using subprocess
output_file_name = "species_equilibrium_out.txt"
database_file = "/Users/cpinilla/software/phreeqc-3.7.3-15968/database/phreeqc.dat"  # Update the path if necessary
phreeqc_executable = "/Users/cpinilla/software/phreeqc-3.7.3-15968/src/phreeqc"  # Use "phreeqc.exe" on Windows, or the full path to the executable

# Run PHREEQC
try:
    subprocess.run([phreeqc_executable, input_file_name, output_file_name, database_file], check=True)
    print(f"PHREEQC run completed. Output saved in '{output_file_name}'.")
except subprocess.CalledProcessError as e:
    print(f"PHREEQC execution failed: {e}")    

# Display the contents of the output file, ignoring problematic characters
try:
    with open(output_file_name, "r", encoding="utf-8", errors="ignore") as output_file:
        output_content = output_file.read()
    print("PHREEQC Output:\n")
    print(output_content)
except FileNotFoundError:
    print(f"Output file '{output_file_name}' not found.")##

#### 2.2. Equilibrio entre Agua y Calcita

El siguiente ejercicio con PHREEQC está diseñado para ayudarte a familiarizarte más con el formato de entrada y salida del programa, calculando la concentración de **Ca²⁺** y **CO₃²⁻** en una solución que se encuentra en **equilibrio con calcita** (CaCO₃).

La primera simulación representa un sistema en el que **agua pura** está en equilibrio con calcita (archivo de entrada 1a). Al revisar los resultados, es importante notar que:

- El **pH cambia** desde un valor inicial de 7 hasta aproximadamente **9.9**.
- Se forman **especies disueltas adicionales**, además de los iones Ca²⁺ y CO₃²⁻, como HCO₃⁻ o complejos calcio-carbonato.

Este tipo de modelo permite estudiar cómo la disolución de un mineral puede modificar la composición química del agua hasta alcanzar condiciones de **saturación**.



In [ ]:
import subprocess

# Step 1: Create the PHREEQC input file
input_file_content = """
TITLE calcite solubility 
SOLUTION 1 pure water & calcite
pH 7
temp 25
EQUILIBRIUM_PHASES
Calcite 0.0 1
END
"""

# Save the input file
input_file_name = "phreeqc_example4.pqi"
with open(input_file_name, "w") as file:
    file.write(input_file_content)
print(f"PHREEQC input file '{input_file_name}' created successfully.")


# Run PHREEQC
try:
    subprocess.run([phreeqc_executable, input_file_name, output_file_name, database_file], check=True)
    print(f"PHREEQC run completed. Output saved in '{output_file_name}'.")
except subprocess.CalledProcessError as e:
    print(f"PHREEQC execution failed: {e}")    

# Display the contents of the output file, ignoring problematic characters
try:
    with open(output_file_name, "r", encoding="utf-8", errors="ignore") as output_file:
        output_content = output_file.read()
    print("PHREEQC Output:\n")
    print(output_content)
except FileNotFoundError:
    print(f"Output file '{output_file_name}' not found.")##

## Problema 2

### Conceptos de especiación acuosa y solubilidad de sólidos y gases

Hemos calculado la concentración de **Ca²⁺** y **CO₃²⁻** en una solución en equilibrio con **calcita** a **25°C**, con un **pH final de aproximadamente 9.9**. Sin embargo, al examinar la salida del modelo, podemos ver que existen otras especies disueltas en la solución que están relacionadas con la ecuación simple que inicialmente se pensaba modelar.

Las interacciones entre el **agua (H₂O)** y la **calcita disuelta** producen varias otras **formas acuosas** de calcio y carbono inorgánico. El ion **CO₃²⁻** está en equilibrio con otras formas disueltas como **HCO₃⁻** y **H₂CO₃**, a través de una serie de reacciones químicas de equilibrio. El conjunto de todas estas especies se denomina **alcalinidad carbonatada** (*carbonate alkalinity*).

Una descripción completa del sistema requiere incluir las siguientes ecuaciones químicas:

$$
\mathrm{CO_2(g)} \leftrightarrow \mathrm{CO_2(aq)}
$$
$$
\mathrm{CO_2(aq)} + \mathrm{H_2O} \leftrightarrow \mathrm{H_2CO_3}
$$
$$
\mathrm{H_2CO_3} \leftrightarrow \mathrm{H^+} + \mathrm{HCO_3^-}
$$
$$
\mathrm{HCO_3^-} \leftrightarrow \mathrm{H^+} + \mathrm{CO_3^{2-}}
$$

Los constantes de equilibrio de estas reacciones se etiquetan generalmente como **KCO₂**, **K₁** y **K₂**, respectivamente. La reacción de disolución del CO₂(g) hasta formar H₂CO₃ suele agruparse como una sola reacción con una constante global **KCO₂**.

Estas ecuaciones, junto con la ecuación de disolución de la calcita, si está presente, permiten describir de forma completa el **sistema carbonato**.

---

### 🔬 ¿Qué es la alcalinidad?

El valor de **alcalinidad** que se reporta en muestras de agua se determina experimentalmente mediante una **valoración con H₂SO₄**, y se expresa en unidades de **meq/L o eq/L de CO₃²⁻, HCO₃⁻ o CaCO₃**.

La alcalinidad corresponde a la suma de:

- HCO₃⁻
- CO₃²⁻
- Otros ácidos débiles presentes como aniones a pH natural (entre 7 y 10), como **boro** o **ácidos orgánicos**.

Se mide como la cantidad de equivalentes de ácido necesarios para disminuir el pH de una muestra de agua (de volumen conocido) hasta **pH 4.5**. El valor de H⁺ se interpreta como **eq/L de alcalinidad**.

En algunos casos, esta información se convierte en **eq/L o meq/L** de HCO₃⁻ y/o CO₃²⁻ usando el valor de pH y asumiendo que la relación HCO₃⁻/CO₃²⁻ depende del pH. Por ejemplo:

$$
10^{-1.3} = \frac{[\mathrm{CO}_3^{2-}] \cdot [\mathrm{H}^+]}{[\mathrm{HCO}_3^-]}
$$

---

### 💭 Pregunta de análisis

Teniendo en cuenta que:

- Un **sistema cerrado** **no** está en equilibrio con el CO₂ atmosférico (por ejemplo agua subterránea),
- Un **sistema abierto** **sí** lo está (por ejemplo, agua superficial expuesta al aire),

**¿Cuál de los dos sistemas disuelve más calcita?**

Para responder esta pregunta:

1. Modela ambos sistemas en PHREEQC (abierto y cerrado al CO₂),
2. Obtén los siguientes parámetros:
   - Concentración de **Ca²⁺**
   - Concentración de **HCO₃⁻**
   - **pH**
   - **Presión parcial de CO₂ (P(CO₂))**

Luego, **compara los resultados** y analiza cuál de los dos permite mayor disolución de calcita.
